# Exercise — Audit & Remediate Catalog Metadata (Customer & Marketing)

**Trailhead Provisions** is under a GDPR audit notice. Using the **AWS Glue Data Catalog**, find
where governance controls are missing for the two domains that hold personal data —
**Customer Identity & Loyalty** and **Marketing & Campaigns** — **fix the gaps in the catalog**,
verify with a re-audit, then write a short reconciliation memo. See `INSTRUCTIONS.md`.

> This exercise runs exclusively against **live AWS Glue** (no local fallback).

In [ ]:
import os
os.environ["CARDINAL_LF_LIVE"] = "1"
os.environ["CARDINAL_AWS_REGION"] = "us-east-1"

import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from governance_toolkit import make_catalog, lf_backend

gc = make_catalog("trailhead.db")
backend = lf_backend()
assert backend == "live", f"Expected live AWS backend, got: {backend}"
print(f"Backend: {backend} -> {type(gc).__name__} (Glue DB: {gc.database}, Region: {gc.region})")

## 1. Run the catalog audit (provided)

In [ ]:
audit = gc.catalog_audit()
print('Total gaps across all domains:', len(audit))
audit

## 2. Narrow to the Customer & Marketing domains
Filter the audit to the two domains you're responsible for.

In [ ]:
my_domains = ["Customer Identity & Loyalty", "Marketing & Campaigns"]
my_gaps = audit[audit["domain"].isin(my_domains)]
my_gaps

## 3. Remediate the gaps in the catalog
Write the missing governance metadata back to Glue with the provided helpers:
`gc.set_table_metadata(table, owner=..., classification=..., retention=...)` and
`gc.tag_column(table, column, classification)` for untagged PII columns.

In [ ]:
# Fix every gap in the two owned domains.
gc.tag_column("customer", "email", "PII")                       # untagged PII column
gc.set_table_metadata("campaign", owner="marketing@trailhead.example")
gc.set_table_metadata("campaign_membership", owner="marketing@trailhead.example",
                      retention="P3Y")                          # ungoverned PII table
gc.tag_column("campaign_membership", "cust_ref", "PII")         # second untagged PII column
print("Remediation written to AWS Glue Data Catalog.")

## 4. Re-audit to verify (provided)
Confirm your two domains are now clean.

In [ ]:
re_audit = gc.catalog_audit()
remaining = re_audit[re_audit["domain"].isin(my_domains)]
print("Gaps in your domains -> before:", len(my_gaps) if my_gaps is not None else "?",
      "after:", len(remaining))
remaining

## 5. Review the cross-domain conflicts (provided)

In [ ]:
gc.semantic_conflicts()

## 6. Write your reconciliation memo
Replace the cell below with your memo. Address every requirement in `INSTRUCTIONS.md`.

### Reconciliation memo — Customer & Marketing domains

**Highest-severity gaps (GDPR-critical), now remediated in the catalog:**
- **`customer.email` was an untagged PII column.** Email is personal data, but it carried no
  classification — invisible to any PII-driven access or subject-access workflow. I tagged it
  `PII`. This is the first thing the GDPR audit would flag.
- **`campaign_membership.cust_ref` was also untagged PII** — it stores the customer's email as
  the marketing join key (a second ungoverned copy of personal data). Tagged `PII`.

**Table-level gaps, now remediated:**
- `campaign` had **no owner** — no accountable steward; set to the marketing domain.
- `campaign_membership` had **no owner and no retention** — ungoverned PII with no deletion
  clock, the worst combination under GDPR; set owner + `P3Y` retention.

**Cross-domain semantic conflicts affecting these domains:**
- **Customer identity key:** loyalty keys customers on `customer_id` (integer) while marketing
  keys on `cust_ref` (email). No shared key, so customer data can only be reconciled across
  domains by matching on email — fragile and a resolution risk.
- **Marketing consent:** `customer.marketing_consent` (loyalty) and
  `campaign_membership.mkt_consent` (marketing) are two systems of record that disagree. Acting
  on the marketing copy when loyalty has withdrawn consent is a direct GDPR violation.

**Recommended standards:** gate publish on `owner`, `classification`, and `retention` for every
table; apply the platform PII tag to personal-data columns at creation; and designate the
loyalty domain as the single consent system of record that marketing reads rather than copies.